#Pythonで学ぶ画像認識　第6章 画像キャプショニング
##第6.3節 CNN-LSTMによる手法〜Show and tellを実装してみよう

###モジュールのインポートとGoogleドライブのマウント

In [2]:
import os
import numpy as np
import datetime
from tqdm import tqdm
import pickle
from typing import Sequence, Dict, Tuple, Union
from collections import deque

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence

import sqlite3

from pydlshogi2.dataloader import HcpeDataLoader
import cshogi
from cshogi.dlshogi import make_input_features, FEATURES1_NUM, FEATURES2_NUM

### 画像エンコーダの実装

In [3]:
from dlshogi.network.policy_value_network import policy_value_network
from dlshogi import serializers
from dlshogi.common import MAX_MOVE_LABEL_NUM
class CNNEncoder(nn.Module):
    '''
    Show and tellのエンコーダ
    dim_embedding: 埋め込み次元
    '''
    def __init__(self, dim_embedding: int, arg_network: str, arg_model: str):
        super().__init__()

        # dlshogiのpolicy networkとvalue networkの出力層前までのネットワークをバックボーンネットワークとする
        model = policy_value_network(arg_network)
        serializers.load_npz(arg_model, model)
        self.backbone = model

        # デコーダへの出力
        self.linear = nn.Linear(2*MAX_MOVE_LABEL_NUM*9*9, dim_embedding)

    '''
    エンコーダの順伝播
    features1: 入力1, [バッチサイズ, チャネル数(62), 高さ(9), 幅(9)]
    features2: 入力2, [バッチサイズ, チャネル数(57), 高さ(9), 幅(9)]
    '''
    def forward(self, features1: torch.Tensor, features2: torch.Tensor):
        # 特徴抽出 -> [バッチサイズ, 2*27(MAX_MOVE_LABEL)*9*9]
        # 今回はバックボーンネットワークは学習させない
        with torch.no_grad():
            policy, value, policy_features, value_features, resnet_features = self.backbone(features1, features2)
            policy_features = torch.flatten(policy_features, 1)
            value_features = torch.flatten(value_features, 1)
            features = torch.cat([policy_features, value_features], dim=-1)

        # 全結合
        features = self.linear(features)

        return features

###文生成デコーダの実装

In [ ]:
class RNNDecoder(nn.Module):
    '''
    Show and tellのデコーダ
    dim_embedding: 埋め込み次元（単語埋め込み次元）
    dim_hidden   : 隠れ層次元
    vocab_size   : 辞書サイズ
    num_layers   : レイヤー数
    dropout      : ドロップアウト確率
    '''
    def __init__(self, dim_embedding: int, dim_hidden: int, 
                 vocab_size: int, num_layers: int, dropout: int=0.1):
        super().__init__()

        # 単語埋め込み
        self.embed = nn.Embedding(vocab_size, dim_embedding)

        # LSTM
        self.lstm = nn.LSTM(dim_embedding, dim_hidden, 
                            num_layers, batch_first=True)

        # 全結合層
        self.linear = nn.Linear(dim_hidden, vocab_size)

        # ドロップアウト
        self.dropout = nn.Dropout(dropout)

    '''
    デコーダの順伝播
    features: エンコーダ出力特徴, [バッチサイズ, 埋め込み次元]
    captions: 画像キャプション,   [バッチサイズ, 系列長]
    lengths : 系列長のリスト
    '''
    def forward(self, features: torch.Tensor, captions: torch.Tensor,
                lengths: list):
        
        # 単語埋め込み -> [バッチサイズ, 系列長, 埋め込み次元]
        embeddings = self.embed(captions)

        # 画像埋め込みと単語埋め込みとを連結
        # features.unsqueeze(1) -> [バッチサイズ, 1, 埋め込み次元]
        # 連結後embeddings -> [バッチサイズ, 系列長 + 1, 埋め込み次元]
        embeddings = torch.cat((features.unsqueeze(1), embeddings), 1)
        
        # パディングされたTensorを可変長系列に戻してパック
        # packed.data() -> [実際の系列長, 埋め込み次元]
        packed = pack_padded_sequence(embeddings,
                                      lengths, batch_first=True)

        # LSTM
        hiddens, cell = self.lstm(packed)

        # ドロップアウト
        output = self.dropout(hiddens[0])

        # ロジットを取得
        outputs = self.linear(output)

        return outputs

    '''
    サンプリングによる説明文出力（貪欲法）
    features  : エンコーダ出力特徴, [バッチサイズ, 埋め込み次元]
    states    : LSTM隠れ状態
    max_length: キャプションの最大系列長
    '''
    @torch.no_grad()
    def sample(self, features, states=None, max_length=30, top_p=0.9):
        inputs = features.unsqueeze(0)  # `[1, batch_size, input_size]` に変換
        word_idx_list = []

        for step_t in range(max_length):
            if states is not None:
                if states[0].dim() == 3 and inputs.dim() == 2:
                    states = (states[0].squeeze(1), states[1].squeeze(1))

            hiddens, states = self.lstm(inputs, states)
            outputs = self.linear(hiddens.squeeze(1))

            # softmax による確率変換
            outputs = outputs.softmax(dim=1)

            # 確率のソート
            sorted_probs, sorted_indices = torch.sort(outputs, descending=True)
            cumulative_probs = torch.cumsum(sorted_probs, dim=1)

            # top_p フィルタリング
            top_p_mask = cumulative_probs > top_p
            top_p_mask[:, 0] = False  # 最も確率が高い要素は削除しない
            sorted_probs[top_p_mask] = 0

            # 確率正規化
            sum_probs = sorted_probs.sum(dim=1, keepdim=True)
            if (sum_probs == 0).all():
                sorted_probs = torch.ones_like(sorted_probs) / sorted_probs.shape[1]  # 一様分布にする
            else:
                sorted_probs /= sum_probs + 1e-8  # 確率正規化

            # サンプリング
            preds = torch.multinomial(sorted_probs, num_samples=1).squeeze()
            selected_word = sorted_indices[0][preds]

            word_idx_list.append(selected_word.item())

            # 次の入力
            inputs = self.embed(selected_word).unsqueeze(0)

        return word_idx_list


###サンプルからミニバッチを生成するcollate関数

In [5]:
'''
batch     : features1, features2, move_label, result, コメントインデックスをまとめたもの
word_to_id: 単語->単語ID辞書
'''
def collate_func(x1, x2, move_label, result, index, word_to_id: Dict[str, int], db_path):

    # SQLiteデータベースに接続
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # index のすべての要素に対して一度に SQL クエリを実行
    index_values = [idx.item() for idx in index]
    placeholders = ', '.join(['?'] * len(index_values))  # SQLクエリ用のプレースホルダ
    cursor.execute(f'SELECT * FROM comments WHERE comment_index IN ({placeholders})', tuple(index_values))
    rows = cursor.fetchall()

    # 取得したデータを使ってトークナイズ
    captions = []
    for row in rows:
        caption = row[1]  # キャプションが row[1] にあると仮定
        captions.append(tokenize_caption(caption, word_to_id))

    # 接続を閉じる
    conn.close()
    
    # キャプションの長さが降順になるように並び替え
    batch = zip(x1, x2, move_label, result, captions)
    batch = sorted(batch, key=lambda x: len(x[4]), reverse=True)
    x1, x2, move_label, result, captions = zip(*batch)
    x1 = torch.stack(x1)
    x2 = torch.stack(x2)
    move_label = torch.stack(move_label)
    result = torch.stack(result)

    lengths = [cap.shape[0] for cap in captions]
    targets = torch.full((len(captions), max(lengths)), 
                         word_to_id['<null>'], dtype=torch.int64)
    for i, cap in enumerate(captions):
        end = lengths[i]
        targets[i, :end] = cap[:end]
    
    return x1, x2, move_label, result, targets, lengths

###トークナイザ

In [6]:
from utils.token_move_tool import words_with_shogi_move
'''
トークナイザ - 文章(caption)を単語IDのリスト(tokens_id)に変換
caption   : 画像キャプション
word_to_id: 単語->単語ID辞書
'''
def tokenize_caption(caption: str, word_to_id: Dict[str, int]):
    tokens = words_with_shogi_move(caption)
    
    tokens_temp = []    
    # 単語についたピリオド、カンマを削除
    for token in tokens:
        if token in {'。', '、', '.', ',', '！', '？', '!', '?'}: 
            continue
        
        tokens_temp.append(token)
    
    tokens = tokens_temp        
        
    # 文章(caption)を単語IDのリスト(tokens_id)に変換
    tokens_ext = ['<start>'] + tokens + ['<end>']
    tokens_id = []
    for k in tokens_ext:
        if k in word_to_id:
            tokens_id.append(word_to_id[k])
        else:
            tokens_id.append(word_to_id['<unk>'])
    
    return torch.Tensor(tokens_id)

###学習におけるハイパーパラメータやオプションの設定

In [6]:
class ConfigTrain(object):
    '''
    ハイパーパラメータ、システム共通変数の設定
    '''  
    def __init__(self):

        # ハイパーパラメータ
        self.dim_embedding = 300 # 埋め込み層の次元
        self.dim_hidden = 128     # LSTM隠れ層の次元
        self.num_layers = 2        # LSTM階層の数
        self.lr = 0.001             # 学習率
        self.dropout = 0.3         # dropout確率
        self.batch_size = 1024       # ミニバッチ数
        self.num_epochs = 100    # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        self.lr_drop = [20]         # 学習率を減衰させるエポック
        
        # パスの設定
        self.train_data = "/workspace/train_AtoR.hcpe"
        self.test_data = "/workspace/test_AtoR.hcpe"
        self.dlshogi_model_path = "/workspace/model/model_resnet10_swish-072_for_caption"
        self.dlshogi_network = "kifcaption"

        self.comment_file = "/workspace/kif_caption/comments_AtoR.db"
        self.word_to_id_file = '/workspace/kif_caption/word_to_id_AtoR.pkl'
        self.save_directory = '/workspace/kif_caption/model'

        # 検証に使う学習セット内のデータの割合
        self.val_ratio = 0.3

        # データローダーに使うCPUプロセスの数
        self.num_workers = 4

        # 学習に使うデバイス
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 移動平均で計算する損失の値の数
        self.moving_avg = 100

### 学習を行う関数

In [7]:
def train():
    config = ConfigTrain()

    # 辞書（単語→単語ID）の読み込み
    with open(config.word_to_id_file, 'rb') as f:
        word_to_id = pickle.load(f)

    # 辞書サイズを保存
    vocab_size = len(word_to_id)
        
    # モデル出力用のディレクトリを作成
    os.makedirs(config.save_directory, exist_ok=True)

    # train dataloader
    train_dataloader = HcpeDataLoader(config.train_data, config.batch_size, config.device, shuffle=True)

    # test data loader
    test_dataloader = HcpeDataLoader(config.test_data, config.batch_size, config.device)

    # モデルの定義
    encoder = CNNEncoder(config.dim_embedding, config.dlshogi_network, config.dlshogi_model_path)
    decoder = RNNDecoder(
        config.dim_embedding, config.dim_hidden, vocab_size, 
        config.num_layers, config.dropout)
    encoder.to(config.device)
    decoder.to(config.device)
    
    # 損失関数の定義
    loss_func = lambda x, y: F.cross_entropy(
        x, y, ignore_index=word_to_id.get('<null>', None))
    
    # 最適化手法の定義
    params = list(decoder.parameters()) \
             + list(encoder.linear.parameters())
    optimizer = torch.optim.AdamW(params, lr=config.lr)

    # 学習率スケジューラの定義
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
                    optimizer, milestones=config.lr_drop, gamma=0.1)
    
    # 学習経過の書き込み
    now = datetime.datetime.now()
    train_loss_file = '{}/6-3_train_loss_{}.csv'\
        .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))
    val_loss_file = '{}/6-3_val_loss_{}.csv'\
        .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))

    total_train_samples = len(train_dataloader)
    total_train_step = total_train_samples // config.batch_size

    total_test_samples = len(test_dataloader)
    total_test_step = total_test_samples // config.batch_size

    # 学習
    val_loss_best = float('inf')
    for epoch in range(config.num_epochs):
        with tqdm(train_dataloader, total=total_train_step) as pbar:
            pbar.set_description(f'[エポック {epoch + 1}]')

            # 学習モードに設定
            encoder.train()
            decoder.train()

            train_losses = deque()
            for x1, x2, move_label, result, index in pbar:
                x1, x2, move_label, result, captions, lengths = collate_func(x1, x2, move_label, result, index, word_to_id, config.comment_file)
                # ミニバッチを設定
                captions = captions.to(config.device)

                optimizer.zero_grad()

                # エンコーダ・デコーダモデル
                features = encoder(x1, x2)
                outputs = decoder(features, captions, lengths)

                # 損失の計算
                targets = pack_padded_sequence(captions, 
                                               lengths, 
                                               batch_first=True)[0]
                loss = loss_func(outputs, targets)

                # 誤差逆伝播
                loss.backward()
                
                optimizer.step()

                # 学習時の損失をログに書き込み
                train_losses.append(loss.item())
                if len(train_losses) > config.moving_avg:
                    train_losses.popleft()
                pbar.set_postfix({
                    'loss': torch.Tensor(train_losses).mean().item()})
                with open(train_loss_file, 'a') as f:
                    print(f'{epoch}, {loss.item()}', file=f)

        # 検証
        with tqdm(test_dataloader, total=total_test_step) as pbar:
            pbar.set_description(f'[検証]')

            # 評価モード
            encoder.eval()
            decoder.eval()

            val_losses = []
            for x1, x2, move_label, result, index in pbar:
                x1, x2, move_label, result, captions, lengths = collate_func(x1, x2, move_label, result, index, word_to_id, config.comment_file)

                # ミニバッチを設定
                captions = captions.to(config.device)

                # エンコーダ-デコーダモデル
                features = encoder(x1, x2)
                outputs = decoder(features, captions, lengths)

                # 損失の計算
                targets = pack_padded_sequence(captions, 
                                               lengths, 
                                               batch_first=True)[0]
                loss = loss_func(outputs, targets)
                val_losses.append(loss.item())

                # Validation Lossをログに書き込み
                with open(val_loss_file, 'a') as f:
                    print(f'{epoch}, {loss.item()}', file=f)

        # Loss 表示
        val_loss = np.mean(val_losses)
        print(f'Validation loss: {val_loss}')

        # より良い検証結果が得られた場合、モデルを保存
        if val_loss < val_loss_best:
            val_loss_best = val_loss

            # エンコーダモデルを保存
            torch.save(
                encoder.state_dict(),
                f'{config.save_directory}/kifcaption_encoder_best.pth')

            # デコーダモデルを保存
            torch.save(
                decoder.state_dict(),
                f'{config.save_directory}/kifcaption_decoder_best.pth')

###学習の実行

In [8]:
train()

[エポック 1]:  13%|█▎        | 73/581 [00:43<04:58,  1.70it/s, loss=7.09]

Error: 'PLACEHOLDER18' 'PLACEHOLDER' in the comment: 「ここで!?」と控室。ただ、▲7七同金△同桂成▲同玉8五桂に▲6六玉が利くのではないかといわれている。△5五角は▲6五玉、△5五金も▲同歩△7七角▲6五玉△5五角成▲7四玉で、竜の力が強い。戻って、△2三金▲6三飛成の交換を入れなければ▲6六玉は△5五角から詰みがあったため指せなかった。


[エポック 1]:  18%|█▊        | 102/581 [01:00<04:44,  1.68it/s, loss=6.67]

Error: 'PLACEHOLDER27' 'PLACEHOLDER' in the comment: 代えて▲5九飛と引くかの2択だった。本譜は5四飛のさばきに重きを置いた手で、現局面からは△8三飛▲7八銀7六角成▲7四飛が進行の一例。


[エポック 1]:  21%|██        | 121/581 [01:11<04:28,  1.71it/s, loss=6.04]

...

Error: 'PLACEHOLDER61' 'PLACEHOLDER' in the comment: この▲2一角が決め手になるか。△同玉は▲2二銀打△同銀▲同銀成△同玉▲2三銀1三玉▲3四銀成で、後手玉が受けなしになる。


[検証]:  78%|███████▊  | 51/65 [00:27<00:07,  1.85it/s]

Error: 'PLACEHOLDER33' 'PLACEHOLDER' in the comment: 上から押さえ込んだ。7五角が動けば△8六歩に▲同角が消える。しかし次の△7五金が甘いため、先手は制約なしに攻めることができる。たとえば▲2六桂3三銀の形は、鈴木流に表現すれば「先手が見切りやすい」。検討陣は先手優勢とみている。


[検証]: 100%|██████████| 65/65 [00:35<00:00,  1.86it/s]

Validation loss: 2.966646227469811


###デモにおけるハイパーパラメータやオプションの設定

In [7]:
class ConfigDemo(object):
    '''
    ハイパーパラメータ、システム共通変数の設定
    '''  
    def __init__(self):

        # ハイパーパラメータ
        self.dim_embedding = 300   # 埋め込み層の次元
        self.dim_hidden = 128      # LSTM隠れ層の次元
        self.num_layers = 2        # LSTM階層の数
        
        # パスの設定

        self.dlshogi_model_path = "/workspace/model/model_resnet10_swish-072_for_caption"
        self.dlshogi_network = "kifcaption"

        self.comment_file = "/workspace/kif_caption/comments_AtoR.db"

        # 画像キャプショニング推論
        # self.img_dirirectory = 'drive/MyDrive/python_image_recognition/data/image_captioning/'    
        self.id_to_word_file = '/workspace/kif_caption/id_to_word_AtoR.pkl'
        self.save_directory = '/workspace/kif_caption/model'
        
        # 推論に使うデバイス
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

###デモを行う関数

In [8]:
def demo(sfen):
    config = ConfigDemo()

    # 辞書（単語ID→単語）の読み込み
    with open(config.id_to_word_file, 'rb') as f:
        id_to_word = pickle.load(f)

    # 辞書サイズを保存
    vocab_size = len(id_to_word)
    
    # エンコーダモデルの定義
    encoder = CNNEncoder(config.dim_embedding, config.dlshogi_network, config.dlshogi_model_path)
    encoder.to(config.device)
    encoder.eval()

    # デコーダモデルの定義
    decoder = RNNDecoder(config.dim_embedding, config.dim_hidden, 
                         vocab_size, config.num_layers)
    decoder.to(config.device)
    decoder.eval()

    # モデルの学習済み重みパラメータをロード
    encoder.load_state_dict(
        torch.load(f'{config.save_directory}/kifcaption_encoder_best.pth'))
    decoder.load_state_dict(
        torch.load(f'{config.save_directory}/kifcaption_decoder_best.pth'))

    # キャプショニング実行

    # 画像読み込み
    board = cshogi.Board(sfen)
    features1 = np.zeros((1, FEATURES1_NUM, 9, 9), dtype=np.float32)
    features2 = np.zeros((1, FEATURES2_NUM, 9, 9), dtype=np.float32)
    make_input_features(board, features1, features2)

    x1 = torch.tensor(features1, device=config.device)
    x2 = torch.tensor(features2, device=config.device)

    # エンコーダ・デコーダモデルによる予測
    feature = encoder(x1, x2)
    sampled_ids = decoder.sample(feature)

    # 入力画像を表示
    print(board)

    # 画像キャプショニングの実行
    sampled_caption = []
    for word_id in sampled_ids:
        word = id_to_word[word_id]
        sampled_caption.append(word)
        if word == '<end>':
            break
    
    sentence = ' '.join(sampled_caption)
    print(f'出力キャプション: {sentence}')

    # 推定結果を書き込み
    gen_sentence_out = "demo_show_and_tell_dlshogi.txt"
    with open(gen_sentence_out, 'w') as f:
        print(sentence, file=f)

###デモの実行

In [10]:
demo(sfen="ln5nl/1r2gg1k1/p3sp1p1/1sp3p1p/1p1p5/2P2P2P/PP1S2PP1/2G3SK1/+bN2RG1NL b BPl2p 1")

'  9  8  7  6  5  4  3  2  1
P1-KY-KE *  *  *  *  * -KE-KY
P2 * -HI *  * -KI-KI * -OU * 
P3-FU *  *  * -GI-FU * -FU * 
P4 * -GI-FU *  *  * -FU * -FU
P5 * -FU * -FU *  *  *  *  * 
P6 *  * +FU *  * +FU *  * +FU
P7+FU+FU * +GI *  * +FU+FU * 
P8 *  * +KI *  *  * +GI+OU * 
P9-UM+KE *  * +HI+KI * +KE+KY
P+00FU
P+00KA
P-00FU00FU
P-00KY
+

出力キャプション: <start> 後手 も 玉 形 を 引き締め た <end>
